In [1]:
# In Silver layer, we are reading the parquet file data from the Bronze layer into dataframes to do the transformations.

In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("silver_layer_cleaning").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/05 06:12:30 WARN Utils: Your hostname, MacBook-Air-105.local, resolves to a loopback address: 127.0.0.1; using 10.0.0.90 instead (on interface en0)
26/03/05 06:12:30 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/05 06:12:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# Reading the parquet data from bronze layer into Dataframes

customers_bronze = spark.read.parquet("/Users/shamilinemmani/Desktop/ecommerce_pipeline/data/bronze/olist_customers_dataset")
orders_bronze = spark.read.parquet("/Users/shamilinemmani/Desktop/ecommerce_pipeline/data/bronze/olist_orders_dataset")
order_items_bronze = spark.read.parquet("/Users/shamilinemmani/Desktop/ecommerce_pipeline/data/bronze/olist_order_items_dataset")
payments_bronze = spark.read.parquet("/Users/shamilinemmani/Desktop/ecommerce_pipeline/data/bronze/olist_order_payments_dataset")


In [4]:
# Transforming Customers Table

In [5]:
customers_bronze.show()

+--------------------+--------------------+------------------------+--------------------+--------------+--------------------+---------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state| ingestion_timestamp|  source_system|
+--------------------+--------------------+------------------------+--------------------+--------------+--------------------+---------------+
|f2a1d75b74d9ec748...|15ee900ec703c9a10...|                   68590|             jacunda|            PA|2026-02-28 08:51:...|olist_ecommerce|
|f15272fe9d0e2ae32...|11e74a9cbe1158d1c...|                   15056|sao jose do rio p...|            SP|2026-02-28 08:51:...|olist_ecommerce|
|7324ecb0ff143f561...|c6be127fa6e30c6f7...|                   13302|                 itu|            SP|2026-02-28 08:51:...|olist_ecommerce|
|7accf3d920f47c07f...|a7f1a6dc9ba06844b...|                   45638|             coaraci|            BA|2026-02-28 08:51:...|olist_ecommerce|
|3680a

In [6]:
customers_bronze.describe().show()

26/03/05 06:12:56 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 7:>                                                          (0 + 1) / 1]

+-------+--------------------+--------------------+------------------------+-------------------+--------------+---------------+
|summary|         customer_id|  customer_unique_id|customer_zip_code_prefix|      customer_city|customer_state|  source_system|
+-------+--------------------+--------------------+------------------------+-------------------+--------------+---------------+
|  count|               99441|               99441|                   99441|              99441|         99441|          99441|
|   mean|                NULL|                NULL|       35137.47458291851|               NULL|          NULL|           NULL|
| stddev|                NULL|                NULL|       29797.93899620613|               NULL|          NULL|           NULL|
|    min|00012a2ce6f8dcda2...|0000366f3b9a7992b...|                    1003|abadia dos dourados|            AC|olist_ecommerce|
|    max|ffffe8b65bbe3087b...|ffffd2657e2aad290...|                   99990|             zortea|        

In [7]:
customers_bronze.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_system: string (nullable = true)



In [8]:
# Checking for duplicates

customers_bronze.selectExpr("count(*) AS total_count", "count(DISTINCT customer_id) AS distinct_customer_count").show()

[Stage 8:=======================================>                   (2 + 1) / 3]

+-----------+-----------------------+
|total_count|distinct_customer_count|
+-----------+-----------------------+
|      99441|                  99441|
+-----------+-----------------------+



In [9]:
# Trim and Capitalize columns

from pyspark.sql.functions import trim, col, initcap, upper
customers_silver = customers_bronze.select(trim(col("customer_id")).alias("customer_id"),
                                          trim(col("customer_unique_id")).alias("customer_unique_id"),
                                           col("customer_zip_code_prefix"),
                                           initcap(trim(col("customer_city"))).alias("customer_city"),
                                           upper(trim(col("customer_state"))).alias("customer_state"),
                                           col("ingestion_timestamp"),
                                           col("source_system"))

In [10]:
customers_silver.show()

[Stage 14:>                                                         (0 + 1) / 1]

+--------------------+--------------------+------------------------+--------------------+--------------+--------------------+---------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state| ingestion_timestamp|  source_system|
+--------------------+--------------------+------------------------+--------------------+--------------+--------------------+---------------+
|f2a1d75b74d9ec748...|15ee900ec703c9a10...|                   68590|             Jacunda|            PA|2026-02-28 08:51:...|olist_ecommerce|
|f15272fe9d0e2ae32...|11e74a9cbe1158d1c...|                   15056|Sao Jose Do Rio P...|            SP|2026-02-28 08:51:...|olist_ecommerce|
|7324ecb0ff143f561...|c6be127fa6e30c6f7...|                   13302|                 Itu|            SP|2026-02-28 08:51:...|olist_ecommerce|
|7accf3d920f47c07f...|a7f1a6dc9ba06844b...|                   45638|             Coaraci|            BA|2026-02-28 08:51:...|olist_ecommerce|
|3680a

In [11]:
# Checking if State has any values not equal to 2 letters

from pyspark.sql.functions import length
customers_silver.filter(length("customer_state") != 2).show()

+-----------+------------------+------------------------+-------------+--------------+-------------------+-------------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|ingestion_timestamp|source_system|
+-----------+------------------+------------------------+-------------+--------------+-------------------+-------------+
+-----------+------------------+------------------------+-------------+--------------+-------------------+-------------+



In [12]:
#Transforming Orders Table

In [13]:
orders_bronze.show()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+---------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date| ingestion_timestamp|  source_system|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+---------------+
|6ec1bea8cbcef0a1b...|e6b5e20566e5c72cb...|   delivered|     2018-08-07 05:42:46|2018-08-07 05:50:08|         2018-08-13 09:24:00|          2018-08-27 15:28:22|          2018-09-18 00:00:00|2026-02-28 08:51:...|olist_ecommerce|
|441972a5bbd51a104...|b79fa9dfed0c3d624...|   delivered|     2018-08-23 22:37:44|2018-08

In [14]:
orders_bronze.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_system: string (nullable = true)



In [15]:
orders_bronze.describe().show()

[Stage 18:==============>                                           (1 + 3) / 4]

+-------+--------------------+--------------------+------------+---------------+
|summary|            order_id|         customer_id|order_status|  source_system|
+-------+--------------------+--------------------+------------+---------------+
|  count|               99441|               99441|       99441|          99441|
|   mean|                NULL|                NULL|        NULL|           NULL|
| stddev|                NULL|                NULL|        NULL|           NULL|
|    min|00010242fe8c5a6d1...|00012a2ce6f8dcda2...|    approved|olist_ecommerce|
|    max|fffe41c64501cc87c...|ffffe8b65bbe3087b...| unavailable|olist_ecommerce|
+-------+--------------------+--------------------+------------+---------------+



In [16]:
# Checking for null primary keys(order_id)

orders_bronze.filter(col("order_id").isNull()).count()

0

In [17]:
# Checking for duplicate order_id

from pyspark.sql.functions import countDistinct
orders_bronze.select(countDistinct("order_id").alias("distinct_order_id")).show()

+-----------------+
|distinct_order_id|
+-----------------+
|            99441|
+-----------------+



In [18]:
# Checking if multiple orders have same customer repeated

orders_bronze.select(countDistinct("customer_id").alias("distinct_customer_id")).show()

+--------------------+
|distinct_customer_id|
+--------------------+
|               99441|
+--------------------+



In [19]:
# Trim and Standardize string columns

from pyspark.sql.functions import trim, lower, col

orders_silver = orders_bronze.select(trim(col("order_id")).alias("order_id"),
                     trim(col("customer_id")).alias("customer_id"),
                     lower(trim(col("order_status"))).alias("order_status"),
                     col("order_purchase_timestamp"),
                     col("order_approved_at"),
                     col("order_delivered_carrier_date"),
                     col("order_delivered_customer_date"),
                     col("order_estimated_delivery_date"),
                     col("ingestion_timestamp"),
                     col("source_system")
                    )
orders_silver.show()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+---------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date| ingestion_timestamp|  source_system|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+---------------+
|6ec1bea8cbcef0a1b...|e6b5e20566e5c72cb...|   delivered|     2018-08-07 05:42:46|2018-08-07 05:50:08|         2018-08-13 09:24:00|          2018-08-27 15:28:22|          2018-09-18 00:00:00|2026-02-28 08:51:...|olist_ecommerce|
|441972a5bbd51a104...|b79fa9dfed0c3d624...|   delivered|     2018-08-23 22:37:44|2018-08

In [20]:
orders_silver.select("order_status").distinct().show()

+------------+
|order_status|
+------------+
|     shipped|
|    canceled|
|    approved|
|    invoiced|
|   delivered|
| unavailable|
|  processing|
|     created|
+------------+



In [21]:
# Validating Timestamp logic

orders_silver.filter(col("order_purchase_timestamp") > col("order_approved_at")).count()


0

In [22]:

orders_silver.filter(col("order_approved_at") > col("order_delivered_customer_date")).count()


61

In [23]:
# Deriving a new column for invalid orders as Order Approval Date is greater than Order Delivered Date

from pyspark.sql.functions import when
orders_silver = orders_silver.withColumn("approval_delivery_valid", when(col("order_approved_at") < col("order_delivered_customer_date"),
                "valid")
                .otherwise("invalid"))
orders_silver.show()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+---------------+-----------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date| ingestion_timestamp|  source_system|approval_delivery_valid|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+---------------+-----------------------+
|6ec1bea8cbcef0a1b...|e6b5e20566e5c72cb...|   delivered|     2018-08-07 05:42:46|2018-08-07 05:50:08|         2018-08-13 09:24:00|          2018-08-27 15:28:22|          2018-09-18 00:00:00|2026-02-28 08:51:...|olist_ecommerce|                 

In [24]:
# Deriving a new column for Total no. of days to deliver

from pyspark.sql.functions import datediff

orders_silver = orders_silver.withColumn("delivery_days", datediff(col("order_delivered_customer_date"), col("order_purchase_timestamp"))
                                        )
orders_silver.show()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+---------------+-----------------------+-------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date| ingestion_timestamp|  source_system|approval_delivery_valid|delivery_days|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+---------------+-----------------------+-------------+
|6ec1bea8cbcef0a1b...|e6b5e20566e5c72cb...|   delivered|     2018-08-07 05:42:46|2018-08-07 05:50:08|         2018-08-13 09:24:00|          2018-08-27 15:28:22|          2018-09-18 00:00:00|2026-02-28 0

In [25]:
# Transforming Order Items Table

In [26]:
order_items_bronze.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_system: string (nullable = true)



In [33]:
order_items_bronze.describe().show()

[Stage 63:==============>                                           (1 + 3) / 4]

+-------+--------------------+------------------+--------------------+--------------------+------------------+------------------+---------------+
|summary|            order_id|     order_item_id|          product_id|           seller_id|             price|     freight_value|  source_system|
+-------+--------------------+------------------+--------------------+--------------------+------------------+------------------+---------------+
|  count|              112650|            112650|              112650|              112650|            112650|            112650|         112650|
|   mean|                NULL|1.1978339991122948|                NULL|                NULL|120.65373901464987|19.990319928982817|           NULL|
| stddev|                NULL|0.7051240313951752|                NULL|                NULL|183.63392805025975|15.806405412297076|           NULL|
|    min|00010242fe8c5a6d1...|                 1|00066f42aeeb9f300...|0015a82c2db000af6...|              0.85|              

In [38]:
order_items_bronze.show()

+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+--------------------+---------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date| price|freight_value| ingestion_timestamp|  source_system|
+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+--------------------+---------------+
|a292ffb436f475f98...|            1|5f21301936c11698d...|fe2032dab1a61af87...|2017-06-29 12:23:42| 149.9|        34.85|2026-02-28 08:52:...|olist_ecommerce|
|a29361cea3ac5ecb5...|            1|5a3320037d5922a77...|b6d44737c04332870...|2018-03-01 15:55:27|  79.0|        24.36|2026-02-28 08:52:...|olist_ecommerce|
|a29361cea3ac5ecb5...|            2|7a6aebc4c1205818e...|8bd0f31cf0a614c65...|2018-03-01 15:55:27|199.99|        26.23|2026-02-28 08:52:...|olist_ecommerce|
|a2940f1b67316eac4...|            1|d0fe4295267f15cca...|0

In [ ]:
# Checking for nulls in order_id & product_id

In [36]:
order_items_bronze.filter(col("product_id").isNull()).count()

0

In [37]:
order_items_bronze.filter(col("order_id").isNull()).count()

0

In [ ]:
# Trimming Columns

In [54]:
order_items_silver = order_items_bronze.select(trim(col("order_id")).alias("order_id"), col("order_item_id"), 
                                               trim(col("product_id")).alias("product_id"), 
                                               trim(col("seller_id")).alias("seller_id"), col("shipping_limit_date"), 
                                               col("price"),
                                               col("freight_value"),
                                               col("ingestion_timestamp"),col("source_system")
)
order_items_silver.show()

+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+--------------------+---------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date| price|freight_value| ingestion_timestamp|  source_system|
+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+--------------------+---------------+
|a292ffb436f475f98...|            1|5f21301936c11698d...|fe2032dab1a61af87...|2017-06-29 12:23:42| 149.9|        34.85|2026-02-28 08:52:...|olist_ecommerce|
|a29361cea3ac5ecb5...|            1|5a3320037d5922a77...|b6d44737c04332870...|2018-03-01 15:55:27|  79.0|        24.36|2026-02-28 08:52:...|olist_ecommerce|
|a29361cea3ac5ecb5...|            2|7a6aebc4c1205818e...|8bd0f31cf0a614c65...|2018-03-01 15:55:27|199.99|        26.23|2026-02-28 08:52:...|olist_ecommerce|
|a2940f1b67316eac4...|            1|d0fe4295267f15cca...|0

In [ ]:
# Deriving a new column and rounding the integer

In [59]:
order_items_silver = order_items_silver.withColumn("total_item_value", round((col("price") + col("freight_value")),2))
order_items_silver.show()

+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+--------------------+---------------+----------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date| price|freight_value| ingestion_timestamp|  source_system|total_item_value|
+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+--------------------+---------------+----------------+
|a292ffb436f475f98...|            1|5f21301936c11698d...|fe2032dab1a61af87...|2017-06-29 12:23:42| 149.9|        34.85|2026-02-28 08:52:...|olist_ecommerce|          184.75|
|a29361cea3ac5ecb5...|            1|5a3320037d5922a77...|b6d44737c04332870...|2018-03-01 15:55:27|  79.0|        24.36|2026-02-28 08:52:...|olist_ecommerce|          103.36|
|a29361cea3ac5ecb5...|            2|7a6aebc4c1205818e...|8bd0f31cf0a614c65...|2018-03-01 15:55:27|199.99|        26.23|2026-02-28 

In [ ]:
# Join Orders with Order items to derive a new column

In [68]:
joined_orders = (
    orders_silver.join(order_items_silver,on = "order_id", how = "inner")
    .withColumn("shipping_limit_days", datediff(col("shipping_limit_date"), col("order_purchase_timestamp")))
)
joined_orders.show()


+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+---------------+-----------------------+-------------+-------------+--------------------+--------------------+-------------------+-----+-------------+--------------------+---------------+----------------+-------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date| ingestion_timestamp|  source_system|approval_delivery_valid|delivery_days|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value| ingestion_timestamp|  source_system|total_item_value|shipping_limit_days|
+--------------------+--------------------+------------+------------------------+-------------------+---------------------------